<a href="https://colab.research.google.com/github/SebGalvaleira/mineria-datos-ii-parcial/blob/main/notebooks/e2/E2_00_setup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# E2 · 00 · Setup del entorno
Instala PySpark, monta Google Drive, trae el dataset desde GitHub y prepara el Data Lake persistente.
Ejecutar al inicio de cada sesión de Colab.

In [1]:
!pip install -q "pyspark>=4.0,<4.1"

import os
os.environ.pop('JAVA_HOME', None)      # que Spark use el Java 21 que trae Colab
!java -version
import pyspark; print('PySpark', pyspark.__version__)

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
openjdk version "21.0.12.1" 2026-08-18
OpenJDK Runtime Environment (build 21.0.12.1+1-1-24.04.4-Ubuntu)
OpenJDK 64-Bit Server VM (build 21.0.12.1+1-1-24.04.4-Ubuntu, mixed mode, sharing)
PySpark 4.0.5


In [2]:
#esto es para poder acceder a drive, de forma que el datalake no se pierda

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
REPO_URL = 'https://github.com/SebGalvaleira/mineria-datos-ii-parcial'
REPO     = Path('/content/repo')
LAKE     = Path('/content/drive/MyDrive/cpa_datalake')     # Data Lake persistente

RUTAS = {z: LAKE / z for z in ['landing', 'stream_input', 'bronze', 'silver',
                               'gold', 'quarantine', 'checkpoints', 'logs']}
for p in RUTAS.values():
    p.mkdir(parents=True, exist_ok=True)
RUTAS

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


{'landing': PosixPath('/content/drive/MyDrive/cpa_datalake/landing'),
 'stream_input': PosixPath('/content/drive/MyDrive/cpa_datalake/stream_input'),
 'bronze': PosixPath('/content/drive/MyDrive/cpa_datalake/bronze'),
 'silver': PosixPath('/content/drive/MyDrive/cpa_datalake/silver'),
 'gold': PosixPath('/content/drive/MyDrive/cpa_datalake/gold'),
 'quarantine': PosixPath('/content/drive/MyDrive/cpa_datalake/quarantine'),
 'checkpoints': PosixPath('/content/drive/MyDrive/cpa_datalake/checkpoints'),
 'logs': PosixPath('/content/drive/MyDrive/cpa_datalake/logs')}

In [3]:
#trae el dataset desde github y copia a landing. Solo copia si Landing está vacía: Landing es inmutable y nunca se pisa.

import shutil, subprocess

if not REPO.exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO)], check=True)

origen = REPO / 'data/datalake/landing'
if not any(RUTAS['landing'].iterdir()):
    shutil.copytree(origen, RUTAS['landing'], dirs_exist_ok=True)
    print('Landing copiada a Drive')
else:
    print('Landing ya existía en Drive: no se toca (es inmutable)')

!ls "{RUTAS['landing']}"
!ls "{RUTAS['landing']}/usage_events_stream" | wc -l

Landing ya existía en Drive: no se toca (es inmutable)
billing_monthly.csv    nps_surveys.csv	    usage_events_stream
customers_orgs.csv     resources.csv	    users.csv
marketing_touches.csv  support_tickets.csv
120


In [4]:
#inicia spark y hace una prueba
from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .appName('cloud-provider-analytics')
         .config('spark.sql.session.timeZone', 'UTC')
         .config('spark.sql.shuffle.partitions', '8')
         .getOrCreate())

df = spark.read.option('header', True).csv(str(RUTAS['landing'] / 'customers_orgs.csv'))
print('Spark', spark.version, '· filas en customers_orgs:', df.count())
df.show(3)

Spark 4.0.5 · filas en customers_orgs: 80
+------------+-------------+---------+----------+---------+-------------+-----------+---------+---------------+----------------+---------+
|      org_id|     org_name| industry| hq_region|plan_tier|is_enterprise|signup_date|sales_rep|lifecycle_stage|marketing_source|nps_score|
+------------+-------------+---------+----------+---------+-------------+-----------+---------+---------------+----------------+---------+
|org_xaji0y6d|Nimbus Labs 0|Education|   sa-east| standard|         True| 2025-05-26|    rep_c|        churned|         partner|     NULL|
|org_pbhsahxt|  Nova Tech 1|Education|   us-east| standard|        False| 2025-06-16|    rep_d|           lead|           event|     -3.0|
|org_hv3a3zmf| Gamma Data 2|    Media|eu-central|      pro|         True| 2025-05-12|    rep_c|         active|         organic|      4.0|
+------------+-------------+---------+----------+---------+-------------+-----------+---------+---------------+-------------